In [1]:
import os
import json
import pandas as pd
from tqdm import tqdm
from transformers import GPT2Tokenizer, GPT2LMHeadModel, Trainer, TrainingArguments
from datasets import load_dataset
import torch

In [2]:
DOMAIN_MAP = {

    # ===== EDUCATION / ACADEMIC =====
    "student": "Education",
    "students": "Education",
    "exam": "Education",
    "grade": "Education",
    "marks": "Education",
    "university": "University",
    "college": "University",
    "faculty": "University",
    "course": "Education",
    "subject": "Education",
    "attendance": "Education",
    "performance": "Education",

    # ===== HEALTH & MEDICAL =====
    "hospital": "Health",
    "patient": "Health",
    "doctor": "Health",
    "medicine": "Health",
    "treatment": "Health",
    "diagnosis": "Health",
    "disease": "Health",
    "clinic": "Health",
    "symptom": "Health",
    "mental": "Mental Health",
    "depression": "Mental Health",
    "anxiety": "Mental Health",

    # ===== RETAIL / E-COMMERCE =====
    "product": "E-Commerce",
    "price": "Sales",
    "sale": "Sales",
    "quantity": "Retail",
    "order": "Retail",
    "cart": "E-Commerce",
    "rating": "Review",
    "review": "Review",
    "feedback": "Review",
    "brand": "Retail",
    "category": "E-Commerce",

    # ===== HOTEL / HOSPITALITY =====
    "hotel": "Hospitality",
    "guest": "Hospitality",
    "booking": "Hospitality",
    "room": "Hospitality",
    "checkin": "Hospitality",
    "checkout": "Hospitality",
    "stay_duration": "Hospitality",
    "restaurant": "Hospitality",

    # ===== AIRLINE / FLIGHT =====
    "flight": "Airline",
    "airline": "Airline",
    "arrival": "Airline",
    "departure": "Airline",
    "airport": "Airline",
    "delay": "Airline",
    "baggage": "Airline",
    "air_cargo": "Airline",

    # ===== VEHICLE / AUTOMOBILE =====
    "vehicle": "Vehicle",
    "car": "Vehicle",
    "bike": "Vehicle",
    "mileage": "Vehicle",
    "engine": "Vehicle",
    "fuel": "Vehicle",
    "disengagement": "Autonomous Driving",
    "automation": "Autonomous Driving",
    "accident": "Crash",
    "crash": "Crash",
    "maintenance": "Vehicle Service",
    "service": "Vehicle Service",
    "comfort": "Vehicle Service",

    # ===== FINANCE / BANKING =====
    "transaction": "Finance",
    "loan": "Finance",
    "credit": "Finance",
    "debit": "Finance",
    "interest": "Finance",
    "balance": "Finance",
    "payment": "Finance",
    "stock": "Finance",
    "investment": "Finance",
    "fraud": "Fraud Detection",
    "insurance": "Insurance",

    # ===== BUSINESS / HR =====
    "employee": "Human Resource",
    "salary": "Employment",
    "pay": "Employment",
    "position": "Human Resource",
    "department": "Human Resource",
    "training": "Human Resource",
    "productivity": "Productivity",

    # ===== MARKETING / SOCIAL MEDIA =====
    "likes": "Social Media",
    "comments": "Social Media",
    "followers": "Social Media",
    "shares": "Social Media",
    "post": "Social Media",
    "campaign": "Marketing",
    "impression": "Marketing",
    "clicks": "Marketing",
    "ad": "Marketing",
    "sentiment": "Marketing",

    # ===== GOVERNMENT / PUBLIC =====
    "crime": "Crime",
    "criminal": "Crime",
    "arrest": "Crime",
    "census": "Demographic",
    "population": "Demographic",
    "pollution": "Environment",
    "waste": "Environment",
    "transport": "Transportation",
    "bus": "Transportation",
    "train": "Transportation",

    # ===== NATURAL DISASTER =====
    "earthquake": "Natural Disaster",
    "richter": "Natural Disaster",
    "tsunami": "Natural Disaster",
    "flood": "Natural Disaster",
    "landslide": "Natural Disaster",
    "cyclone": "Natural Disaster",
    "hurricane": "Natural Disaster",
    "disaster": "Natural Disaster",

    # ===== WEATHER / CLIMATE =====
    "temperature": "Climate",
    "humidity": "Climate",
    "wind": "Climate",
    "rainfall": "Climate",
    "weather": "Climate",

    # ===== AGRICULTURE =====
    "crop": "Agriculture",
    "yield": "Agriculture",
    "soil": "Agriculture",
    "fertilizer": "Agriculture",
    "livestock": "Agriculture",

    # ===== MANUFACTURING / INDUSTRY =====
    "factory": "Manufacturing",
    "machine": "Manufacturing",
    "production": "Manufacturing",
    "defect": "Manufacturing",
    "quality": "Manufacturing",
    "inventory": "Supply Chain",
    "warehouse": "Supply Chain",
    "shipment": "Supply Chain",
    "supplier": "Supply Chain",

    # ===== IT / SOFTWARE / CYBERSECURITY =====
    "server": "IT",
    "api": "IT",
    "endpoint": "IT",
    "latency": "IT",
    "crash_log": "IT",
    "requests": "IT",
    "bug": "Software",
    "error": "Software",
    "attack": "Cybersecurity",
    "threat": "Cybersecurity",

    # ===== ENERGY & UTILITIES =====
    "energy": "Energy",
    "electricity": "Energy",
    "gas": "Energy",
    "meter": "Energy",
    "consumption": "Energy",

    # ===== REAL ESTATE =====
    "property": "Real Estate",
    "house": "Real Estate",
    "rent": "Real Estate",
    "mortgage": "Real Estate",

    # ===== LOGISTICS =====
    "delivery": "Logistics",
    "route": "Logistics",
    "distance": "Logistics",
    "fleet": "Logistics",
}


In [3]:
def create_professional_title(domain):
    domain = domain.lower()

    # ===== EDUCATION / ACADEMIC =====
    if "student" in domain or "education" in domain:
        return "Comprehensive Analysis of Student Academic Performance and Learning Outcomes"

    if "university" in domain:
        return "University Academic Efficiency and Institutional Performance Evaluation"

    if "course" in domain:
        return "Course-Level Assessment and Curriculum Effectiveness Study"

    if "attendance" in domain:
        return "Student Attendance Patterns and Academic Behavior Insights"

    # ===== HEALTH & MEDICAL =====
    if "hospital" in domain or "health" in domain:
        return "Hospital Patient Record Analysis and Healthcare Performance Review"

    if "clinical" in domain:
        return "Clinical Trial Effectiveness and Medical Response Evaluation"

    if "medical" in domain:
        return "Medical Treatment Effectiveness and Patient Outcome Study"

    if "mental" in domain:
        return "Mental Health Indicators and Psychological Well-Being Analysis"

    # ===== RETAIL / E-COMMERCE =====
    if "e-commerce" in domain or "product" in domain:
        return "E-Commerce Product Performance and Customer Purchasing Behavior Report"

    if "retail" in domain:
        return "Retail Sales Insights and Store Performance Optimization Study"

    if "refund" in domain:
        return "Refund and Return Pattern Analysis in Customer Transactions"

    if "cart" in domain:
        return "Shopping Cart Abandonment Insights and Customer Motivation Study"

    # ===== HOTEL / HOSPITALITY =====
    if "hotel" in domain or "hospitality" in domain:
        return "Hotel Guest Satisfaction and Hospitality Service Experience Evaluation"

    if "restaurant" in domain:
        return "Restaurant Menu Performance and Customer Dining Experience Study"

    # ===== AIRLINE / TRANSPORT =====
    if "airline" in domain or "flight" in domain:
        return "Airline Flight Performance, Delay Metrics, and Passenger Experience Review"

    if "airport" in domain:
        return "Airport Passenger Traffic and Flight Operations Analysis"

    if "air cargo" in domain:
        return "Air Cargo Logistics Efficiency and Transportation Flow Report"

    # ===== AUTOMOBILE / VEHICLE =====
    if "vehicle service" in domain:
        return "Vehicle Service Quality, Maintenance Efficiency, and Comfort Evaluation"

    if "vehicle" in domain:
        return "Vehicle Usage Patterns and Transportation System Performance Summary"

    if "autonomous" in domain or "disengagement" in domain:
        return "Autonomous Vehicle Disengagement Statistics and Safety Performance Analysis"

    if "crash" in domain:
        return "Vehicle Crash Analysis and Road Safety Risk Assessment"

    # ===== FINANCE / BANKING =====
    if "finance" in domain:
        return "Financial Performance, Market Indicators, and Economic Trend Evaluation"

    if "loan" in domain:
        return "Loan Risk Assessment and Borrower Financial Health Report"

    if "investment" in domain:
        return "Investment Portfolio Performance and Market Trend Analysis"

    if "fraud" in domain:
        return "Financial Fraud Detection and Transaction Integrity Assessment"

    if "insurance" in domain:
        return "Insurance Claim Analysis and Risk Evaluation Report"

    # ===== BUSINESS / HR =====
    if "employment" in domain:
        return "Employment Structure, Salary Distribution, and Workforce Analysis"

    if "human resource" in domain:
        return "Employee Performance Review and HR Operational Efficiency Study"

    if "productivity" in domain:
        return "Workplace Productivity Metrics and Organizational Efficiency Evaluation"

    # ===== MARKETING / SOCIAL MEDIA =====
    if "marketing" in domain:
        return "Marketing Campaign Effectiveness and Customer Engagement Analysis"

    if "social media" in domain:
        return "Social Media Engagement Metrics and Audience Behavior Insights"

    if "advertising" in domain:
        return "Advertising Response Patterns and Campaign Impact Evaluation"

    if "brand" in domain:
        return "Brand Sentiment Analysis and Customer Perception Tracking"

    # ===== GOVERNMENT / PUBLIC DATA =====
    if "crime" in domain:
        return "Crime Rate Patterns and Public Safety Assessment Report"

    if "census" in domain or "demographic" in domain:
        return "Population Demographic Structure and Societal Distribution Analysis"

    if "pollution" in domain or "environment" in domain:
        return "Environmental Pollution Trends and Ecological Impact Assessment"

    if "transportation" in domain:
        return "Public Transportation Usage Patterns and Operational Efficiency Study"

    if "water" in domain:
        return "Water Quality Assessment and Resource Distribution Analysis"

    # ===== SCIENCE / NATURAL DISASTERS =====
    if "earthquake" in domain:
        return "Earthquake Seismic Activity Evaluation and Impact Assessment"

    if "tsunami" in domain:
        return "Tsunami Warning Indicators and Coastal Risk Vulnerability Report"

    if "flood" in domain:
        return "Flood Risk Analysis and Hydrological Impact Assessment"

    if "climate" in domain:
        return "Climate Trend Analysis and Environmental Condition Monitoring"

    # ===== AGRICULTURE =====
    if "agriculture" in domain:
        return "Crop Yield Optimization and Agricultural Productivity Analysis"

    if "soil" in domain:
        return "Soil Quality Assessment and Fertility Trend Evaluation"

    if "livestock" in domain:
        return "Livestock Health Monitoring and Farm Productivity Summary"

    # ===== MANUFACTURING / INDUSTRY =====
    if "manufacturing" in domain:
        return "Manufacturing Line Efficiency and Industrial Quality Assessment"

    if "machine failure" in domain:
        return "Machine Failure Prediction and Industrial Maintenance Insights"

    if "supply chain" in domain:
        return "Supply Chain Performance and Inventory Optimization Report"

    if "warehouse" in domain:
        return "Warehouse Efficiency and Storage Management Analysis"

    # ===== IT / SOFTWARE / CYBERSECURITY =====
    if "software" in domain:
        return "Software Application Usage and Stability Insight Report"

    if "server" in domain:
        return "Server Performance Metrics and System Load Analysis"

    if "cybersecurity" in domain:
        return "Cybersecurity Threat Detection and Risk Mitigation Assessment"

    if "api" in domain:
        return "API Performance Monitoring and Response Time Evaluation"

    # ===== ENERGY & UTILITIES =====
    if "energy" in domain:
        return "Energy Consumption Patterns and Utility Efficiency Analysis"

    if "electricity" in domain:
        return "Electricity Usage Trends and Power Grid Performance Study"

    if "pipeline" in domain:
        return "Pipeline Flow Analytics and Resource Distribution Monitoring"

    # ===== REAL ESTATE =====
    if "real estate" in domain:
        return "Real Estate Market Analysis and Property Value Trend Evaluation"

    if "rental" in domain:
        return "Rental Demand Forecast and Housing Occupancy Trend Report"

    # ===== LOGISTICS / SUPPLY =====
    if "logistics" in domain:
        return "Logistics Network Efficiency and Delivery Performance Evaluation"

    if "delivery" in domain:
        return "Delivery Time Optimization and Shipment Accuracy Study"

    # ===== DEFAULT =====
    return "Comprehensive Dataset Trend and Pattern Evaluation"


In [4]:
def summarize_csv(file_path):
    # Try UTF-8, fallback to ISO
    try:
        df = pd.read_csv(file_path, engine="python", on_bad_lines="skip")
    except:
        df = pd.read_csv(file_path, engine="python", encoding="ISO-8859-1", on_bad_lines="skip")

    if df.empty:
        return None, []

    cols = df.columns.tolist()

    summary = f"Columns: {', '.join(cols)}. Rows: {len(df)}. "

    numeric_cols = df.select_dtypes(include='number').columns
    if len(numeric_cols) > 0:
        summary += f"Contains numeric fields such as: {', '.join(numeric_cols)}. "
    else:
        summary += "No numeric fields detected."

    return summary.strip(), cols


In [5]:
def detect_domain(summary, cols):
    text = (summary + " " + " ".join(cols)).lower()

    for keyword, domain in DOMAIN_MAP.items():
        if keyword in text:
            return domain

    return "General"


In [ ]:
input_folder = "data"
output_file = "data/summaries.jsonl"

os.makedirs("data", exist_ok=True)

final_data = []

for file in tqdm(os.listdir(input_folder)):
    if file.endswith(".csv"):
        path = os.path.join(input_folder, file)

        summary, cols = summarize_csv(path)
        if summary is None:
            continue

        domain = detect_domain(summary, cols)
        title = create_professional_title(domain)

        final_data.append({"csv_data_summary": summary, "title": title})

with open(output_file, "w", encoding="utf-8") as f:
    for item in final_data:
        f.write(json.dumps(item) + "\n")

print("✅ Training dataset created at:", os.path.abspath(output_file))


100%|████████████████████████████████████████████████████████████████████████████████| 471/471 [01:02<00:00,  7.49it/s]

✅ Training dataset created at: C:\Users\lenovo\data\summaries.jsonl


In [7]:
tokenizer = GPT2Tokenizer.from_pretrained("distilgpt2")
tokenizer.pad_token = tokenizer.eos_token

model = GPT2LMHeadModel.from_pretrained("distilgpt2")

dataset = load_dataset("json", data_files="data/summaries.jsonl", split="train")


Generating train split: 0 examples [00:00, ? examples/s]

In [8]:
def preprocess(batch):
    texts = [f"Summary: {s}\nTitle: {t}" for s, t in zip(batch["csv_data_summary"], batch["title"])]

    output = tokenizer(
        texts,
        max_length=256,
        padding="max_length",
        truncation=True,
        return_tensors="np"
    )

    output["labels"] = output["input_ids"]
    return output

tokenized = dataset.map(preprocess, batched=True, remove_columns=dataset.column_names)


Map:   0%|          | 0/466 [00:00<?, ? examples/s]

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)

training_args = TrainingArguments(
    output_dir="./gpt2-title-model",
    per_device_train_batch_size=2,
    num_train_epochs=8,
    learning_rate=5e-5,
    save_strategy="epoch",
    logging_dir="./logs",
    dataloader_pin_memory=False  # removes GPU warning
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized
)

trainer.train()

trainer.save_model("./model")
tokenizer.save_pretrained("./model")

print("🎉 TRAINING COMPLETE!")

# ============================
# SAVE FINAL CHECKPOINT
# ============================
FINAL_MODEL_DIR = "./model"

import os
os.makedirs(FINAL_MODEL_DIR, exist_ok=True)

model.save_pretrained(FINAL_MODEL_DIR)
tokenizer.save_pretrained(FINAL_MODEL_DIR)

print("✅ Final checkpoint saved at:", os.path.abspath(FINAL_MODEL_DIR))


`loss_type=None` was set in the config but it is unrecognised.Using the default loss: `ForCausalLMLoss`.


Step,Training Loss
500,0.721800
1000,0.452500
1500,0.371700


🎉 TRAINING COMPLETE!
✅ Final checkpoint saved at: C:\Users\lenovo\title_model_checkpoint


In [10]:
def clean_generated_title(text):
    if "Title:" in text:
        text = text.split("Title:", 1)[1]
    return text.split("\n")[0].strip()


In [11]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)

def generate_title_from_csv(file_path):
    summary, _ = summarize_csv(file_path)

    prompt = f"Summary: {summary}\nTitle:"

    inputs = tokenizer(prompt, return_tensors="pt", padding=True, truncation=True)

    # MOVE INPUTS TO GPU
    inputs = {k: v.to(device) for k, v in inputs.items()}

    output = model.generate(
        inputs["input_ids"],
        max_new_tokens=35,
        do_sample=True,
        temperature=0.7,
        pad_token_id=tokenizer.eos_token_id
    )

    decoded = tokenizer.decode(output[0], skip_special_tokens=True)
    return clean_generated_title(decoded)

In [ ]:
file_path = "data\2019AutonomousVehicleDisengagementReports.csv"
print("Generated Title:", generate_title_from_csv(file_path))

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Generated Title: Comprehensive Dataset Trend and Pattern Evaluation


In [13]:
import os
print(os.path.abspath("./gpt2-title-model"))


C:\Users\lenovo\gpt2-title-model
